# บทเรียนที่ 13 - ความทรงจำของเอเจนต์ด้วยกราฟความรู้ Cognee


## การติดตั้ง

สมุดบันทึกนี้สาธิตวิธีการสร้าง **ผู้ช่วยเขียนโค้ด** อัจฉริยะที่มีหน่วยความจำถาวรโดยใช้กราฟความรู้ [**Cognee**](https://www.cognee.ai/) และ **Microsoft Agent Framework** (MAF)

Cognee แปลงข้อความที่ไม่มีโครงสร้างให้กลายเป็นกราฟความรู้ที่มีโครงสร้างและสามารถสืบค้นได้ ซึ่งสนับสนุนด้วยเวกเตอร์ฝังตัว — มอบหน่วยความจำระยะยาวที่มีความสัมพันธ์และสมบูรณ์ให้กับตัวแทนของคุณ

### สิ่งที่คุณจะได้เรียนรู้
1. **สร้างกราฟความรู้**: แปลงโปรไฟล์นักพัฒนาและแนวทางปฏิบัติที่ดีที่สุดให้เป็นความรู้ที่มีโครงสร้างและสามารถสืบค้นได้
2. **รวม Cognee กับ MAF**: ใช้ฟังก์ชัน `@tool` เพื่อให้ตัวแทน MAF สืบค้นกราฟความรู้ของ Cognee
3. **บทสนทนาที่รับรู้เซสชัน**: รักษาบริบทข้ามคำถามหลายๆ คำถามในเซสชันเดียวกัน
4. **หน่วยความจำระยะยาว**: เก็บรักษาความรู้ที่สำคัญข้ามเซสชันและดึงมาใช้ในการสนทนาใหม่ๆ

### สิ่งที่ต้องมี
- Python 3.9+
- Redis ที่รันในเครื่อง (`docker run -d -p 6379:6379 redis`) สำหรับการจัดการเซสชัน
- คีย์ API ของ LLM (เช่น OpenAI) — ตั้งค่า `LLM_API_KEY` ใน `.env`
- ตั้งค่า `CACHING=true` ใน `.env` (จำเป็นสำหรับเซสชัน Cognee)
- โครงการ Azure AI Foundry ที่มีโมเดลแชทที่ปรับใช้แล้ว
- ตั้งค่า `AZURE_AI_PROJECT_ENDPOINT` และ `AZURE_AI_MODEL_DEPLOYMENT_NAME` ใน `.env`
- ทำการยืนยันตัวตน Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ประเภทของหน่วยความจำตัวแทน

สมุดบันทึกนี้สำรวจสามประเภทของหน่วยความจำเดียวกันจากสมุดบันทึกบทเรียนหลักบทที่ 13 แต่ใช้ Cognee เป็นระบบหน่วยความจำระยะยาว:

| ประเภทของหน่วยความจำ | กลไก | อายุการใช้งาน |
|---|---|---|
| **ทำงาน** | `agent.create_session()` (MAF) | การสนทนาเพียงครั้งเดียว |
| **ระยะสั้น** | แคชเซสชัน Cognee (Redis) | เซสชันหนึ่งครั้ง |
| **ระยะยาว** | กราฟความรู้ Cognee + เวกเตอร์ | ถาวร |

### สถาปัตยกรรมหน่วยความจำของ Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## เตรียมการจัดเก็บ Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Part 1 — การสร้างฐานความรู้

เรารับข้อมูลสามประเภทเพื่อสร้างฐานความรู้ที่ครอบคลุมสำหรับผู้ช่วยโค้ดของเรา:

1. **โปรไฟล์นักพัฒนา** — ความเชี่ยวชาญส่วนตัวและพื้นฐานทางเทคนิค
2. **แนวทางปฏิบัติที่ดีที่สุดของ Python** — Zen of Python พร้อมแนวทางปฏิบัติ
3. **บทสนทนาในอดีต** — ชุดถามตอบระหว่างนักพัฒนาและผู้ช่วย AI ในอดีต


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualize the Knowledge Graph

Cognee สามารถแสดงภาพแผนภูมิความรู้เชิงโต้ตอบในรูปแบบ HTML ของเอนทิตีและความสัมพันธ์ที่มันดึงออกมาได้


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## เติมเต็มความทรงจำด้วย Memify

`memify()` วิเคราะห์กราฟความรู้และสร้างกฎอัจฉริยะ — ระบุรูปแบบ แนวทางปฏิบัติที่ดีที่สุด และความสัมพันธ์ระหว่างแนวคิดต่างๆ


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## ส่วนที่ 2 — ตัวแทน MAF ที่มีเครื่องมือ Cognee

ตอนนี้เราสร้างตัวแทน MAF ที่สามารถสอบถามกราฟความรู้ของ Cognee ผ่านฟังก์ชัน `@tool` ฟังก์ชันนี้ช่วยให้ตัวแทนสามารถใช้ประโยชน์เต็มที่จากการค้นหาเชิงความหมายตามกราฟได้ในขณะที่ยังคงรักษาบริบทการสนทนาผ่านเซสชันไว้ได้อีกด้วย


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## หน่วยความจำการทำงานกับเซสชัน

`AgentSession` (สร้างขึ้นผ่าน `agent.create_session()`) ให้หน่วยความจำการทำงานภายในเซสชัน ตัวแทนสามารถอ้างอิงกลับไปยังข้อความก่อนหน้าในขณะเดียวกันก็สามารถสืบค้นกราฟความรู้ระยะยาวของ Cognee ได้ด้วยเช่นกัน


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## เซสชันใหม่ — ความจำระยะยาวยังคงอยู่

การเริ่มเซสชันใหม่จะล้างหน่วยความจำการทำงาน แต่กราฟความรู้ Cognee ยังคงใช้ได้ ตัวแทนสามารถดึงข้อมูลความรู้ระยะยาวเดียวกันในการสนทนาใหม่ทั้งหมดได้


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## สรุป

ในสมุดบันทึกนี้คุณได้สร้างผู้ช่วยเขียนโค้ดที่รวม **หน่วยความจำทำงานของ MAF** (`agent.create_session()`) กับ **กราฟความรู้ระยะยาวของ Cognee** 

### สิ่งที่คุณได้เรียนรู้
1. **การสร้างกราฟความรู้**: Cognee รับข้อความที่ไม่มีโครงสร้างและสร้างกราฟ + หน่วยความจำเวกเตอร์
2. **การเติมเต็มกราฟด้วย memify**: ข้อเท็จจริงที่ได้มาและความสัมพันธ์ที่ลึกซึ้งยิ่งขึ้นบนกราฟที่มีอยู่
3. **การรวม MAF + Cognee**: ฟังก์ชัน `@tool` ช่วยให้เอเจนต์ MAF สืบค้นกราฟของ Cognee ได้อย่างเป็นธรรมชาติ
4. **หน่วยความจำทำงาน + หน่วยความจำระยะยาว**: `AgentSession` (ผ่าน `agent.create_session()`) ให้บริบทของเซสชันในขณะที่ Cognee ให้ความรู้ที่คงทน
5. **การค้นหาแบบกรองด้วย NodeSets**: กำหนดเป้าหมายเฉพาะบางส่วนของกราฟความรู้ (เช่น เฉพาะหลักการ)

### ข้อคิดสำคัญ
- **Cognee** เปลี่ยนข้อความดิบให้กลายเป็นหน่วยความจำที่มีโครงสร้างและตระหนักถึงความสัมพันธ์ — มีพลังมากกว่าการจัดเก็บเวกเตอร์แบบง่าย
- **ฟังก์ชัน `@tool`** เชื่อมต่อเอเจนต์ MAF กับระบบความรู้ภายนอกอย่างสะอาดและเป็นระเบียบ
- **`AgentSession`** (ผ่าน `agent.create_session()`) เก็บบริบทต่อบทสนทนาแยกจากความรู้ที่อยู่ยาวนาน
- กราฟความรู้เดียวกันบริการหลายเซสชันและหลายเอเจนต์ได้พร้อมกัน

### การประยุกต์ใช้งานในโลกจริง
- **ผู้ช่วยเขียนโค้ดสำหรับนักพัฒนา**: ตรวจสอบโค้ด วิเคราะห์เหตุการณ์ ผู้ช่วยสถาปัตยกรรม
- **ผู้ช่วยที่ติดต่อกับลูกค้า**: เอเจนต์สนับสนุนผ่านเอกสารผลิตภัณฑ์, คำถามที่พบบ่อย, และบันทึก CRM
- **ผู้ช่วยผู้เชี่ยวชาญภายในองค์กร**: ผู้ช่วยด้านนโยบาย, กฎหมาย, หรือความปลอดภัย วิเคราะห์ข้อกำหนด
- **เลเยอร์ข้อมูลแบบรวมศูนย์**: รวมข้อมูลที่มีโครงสร้างและไม่มีโครงสร้างไว้ในกราฟเดียวที่สามารถสืบค้นได้

### ขั้นตอนถัดไป
- ทดลองความรับรู้เชิงเวลาของ Cognee
- กำหนดออนโทโลยี OWL สำหรับคุณภาพกราฟเฉพาะโดเมน
- เพิ่มวงจรตอบรับจากผู้ใช้เพื่อปรับปรุงการดึงข้อมูลในระยะยาว
- ขยายสู่ระบบหลายเอเจนต์ที่แชร์เลเยอร์ความจำ Cognee เดียวกัน


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:  
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้ความถูกต้องสูงสุด แต่โปรดทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่แม่นยำ เอกสารต้นฉบับในภาษาดั้งเดิมถือเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ ขอแนะนำให้ใช้บริการแปลโดยผู้เชี่ยวชาญมืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดใด ๆ ที่เกิดจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
